<a href="https://colab.research.google.com/github/purnimakushwaha/ITC101_Minor-project_python/blob/main/Random_fact_explorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 🌍 RANDOM FACT EXPLORER
# API + JSON + Pandas + Matplotlib + ipywidgets
# ============================================================

import requests
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import os
import time

from datetime import datetime
from IPython.display import display, HTML, clear_output


# ============================================================
# SETTINGS
# ============================================================

API_URL = "https://uselessfacts.jsph.pl/api/v2/facts/random"

HISTORY_FILE = "fact_history.csv"


# ============================================================
# GLOBAL VARIABLES
# ============================================================

current_fact = ""

fact_history = []


# ============================================================
# LOAD PREVIOUS HISTORY
# ============================================================

if os.path.exists(HISTORY_FILE):

    try:

        history_df = pd.read_csv(
            HISTORY_FILE
        )

    except Exception:

        history_df = pd.DataFrame(
            columns=[
                "Date",
                "Time",
                "Fact",
                "Source"
            ]
        )

else:

    history_df = pd.DataFrame(
        columns=[
            "Date",
            "Time",
            "Fact",
            "Source"
        ]
    )


# ============================================================
# HEADER
# ============================================================

header = widgets.HTML(
    """
    <div style="
        background:#6a1b9a;
        color:white;
        padding:25px;
        border-radius:18px;
        text-align:center;
        font-family:Arial;
    ">

        <div style="
            font-size:32px;
            font-weight:bold;
        ">
            🌍 RANDOM FACT EXPLORER
        </div>

        <div style="
            margin-top:8px;
            font-size:15px;
        ">
            Discover Interesting Facts Using an API
        </div>

    </div>
    """
)


# ============================================================
# BUTTONS
# ============================================================

new_fact_button = widgets.Button(
    description="🔄 New Fact",
    button_style="primary"
)


favorite_button = widgets.Button(
    description="⭐ Favorite",
    button_style="warning"
)


history_button = widgets.Button(
    description="📋 History",
    button_style="info"
)


statistics_button = widgets.Button(
    description="📊 Statistics",
    button_style="success"
)


clear_history_button = widgets.Button(
    description="🗑️ Clear History",
    button_style="danger"
)


# ============================================================
# OUTPUT AREA
# ============================================================

output = widgets.Output()


# ============================================================
# STATUS AREA
# ============================================================

status_output = widgets.Output()


# ============================================================
# FETCH FACT FROM API
# ============================================================

def fetch_fact():

    global current_fact


    start_time = time.perf_counter()


    response = requests.get(
        API_URL,
        timeout=10
    )


    api_time = (
        time.perf_counter()
        - start_time
    )


    response.raise_for_status()


    data = response.json()


    current_fact = data.get(
        "text",
        "No fact found."
    )


    source = data.get(
        "source",
        "Unknown"
    )


    return current_fact, source, api_time


# ============================================================
# SHOW NEW FACT
# ============================================================

def show_new_fact(
    button=None
):

    try:

        with output:

            clear_output()


            display(
                HTML(
                    """
                    <h3>
                    🔄 Fetching a new fact...
                    </h3>
                    """
                )
            )


        fact, source, api_time = fetch_fact()


        # ----------------------------------------------------
        # DISPLAY FACT
        # ----------------------------------------------------

        with output:

            clear_output()


            display(
                HTML(
                    f"""
                    <div style="
                        background:#f3e5f5;
                        padding:30px;
                        border-radius:20px;
                        border:2px solid #8e24aa;
                        text-align:center;
                        margin-top:20px;
                    ">

                        <div style="
                            font-size:50px;
                        ">
                            💡
                        </div>

                        <h2>
                            DID YOU KNOW?
                        </h2>

                        <p style="
                            font-size:20px;
                            line-height:1.6;
                        ">
                            {fact}
                        </p>

                        <hr>

                        <p style="
                            font-size:13px;
                            color:#666;
                        ">
                            🌐 Source:
                            {source}
                        </p>

                        <p style="
                            font-size:13px;
                            color:#666;
                        ">
                            ⚡ API Response Time:
                            {api_time:.3f} seconds
                        </p>

                    </div>
                    """
                )
            )


        # ----------------------------------------------------
        # SAVE TO HISTORY
        # ----------------------------------------------------

        save_fact_to_history(
            fact,
            source
        )


    except Exception as error:

        with output:

            clear_output()


            display(
                HTML(
                    f"""
                    <div style="
                        background:#ffebee;
                        padding:20px;
                        border-radius:12px;
                    ">

                    <h3>
                    ❌ Unable to Fetch Fact
                    </h3>

                    <p>
                    {error}
                    </p>

                    <p>
                    Please check your internet
                    connection.
                    </p>

                    </div>
                    """
                )
            )


# ============================================================
# SAVE FACT TO HISTORY
# ============================================================

def save_fact_to_history(
    fact,
    source
):

    global history_df


    now = datetime.now()


    new_record = {

        "Date":
            now.strftime(
                "%Y-%m-%d"
            ),

        "Time":
            now.strftime(
                "%H:%M:%S"
            ),

        "Fact":
            fact,

        "Source":
            source
    }


    history_df.loc[
        len(history_df)
    ] = new_record


    history_df.to_csv(
        HISTORY_FILE,
        index=False
    )


# ============================================================
# FAVORITE FACT
# ============================================================

def favorite_fact(
    button=None
):

    if not current_fact:

        with status_output:

            clear_output()


            display(
                HTML(
                    """
                    <p>
                    ⚠️ Generate a fact first.
                    </p>
                    """
                )
            )

        return


    favorite_file = "favorite_facts.csv"


    if os.path.exists(
        favorite_file
    ):

        favorites_df = pd.read_csv(
            favorite_file
        )

    else:

        favorites_df = pd.DataFrame(
            columns=[
                "Date",
                "Time",
                "Fact"
            ]
        )


    now = datetime.now()


    new_favorite = {

        "Date":
            now.strftime(
                "%Y-%m-%d"
            ),

        "Time":
            now.strftime(
                "%H:%M:%S"
            ),

        "Fact":
            current_fact
    }


    favorites_df.loc[
        len(favorites_df)
    ] = new_favorite


    favorites_df.to_csv(
        favorite_file,
        index=False
    )


    with status_output:

        clear_output()


        display(
            HTML(
                """
                <div style="
                    background:#fff3cd;
                    padding:15px;
                    border-radius:12px;
                ">

                ⭐ Fact added to favorites!

                </div>
                """
            )
        )


# ============================================================
# SHOW HISTORY
# ============================================================

def show_history(
    button=None
):

    with output:

        clear_output()


        if history_df.empty:

            display(
                HTML(
                    """
                    <h3>
                    📋 No fact history available.
                    </h3>
                    """
                )
            )

            return


        display(
            HTML(
                """
                <h2>
                📋 Fact History
                </h2>
                """
            )
        )


        display(
            history_df
        )


# ============================================================
# SHOW STATISTICS
# ============================================================

def show_statistics(
    button=None
):

    with output:

        clear_output()


        if history_df.empty:

            display(
                HTML(
                    """
                    <h3>
                    📊 No data available.
                    </h3>
                    """
                )
            )

            return


        total_facts = len(
            history_df
        )


        unique_sources = (
            history_df[
                "Source"
            ]
            .nunique()
        )


        display(
            HTML(
                f"""
                <div style="
                    background:#e8f5e9;
                    padding:25px;
                    border-radius:18px;
                ">

                    <h2>
                    📊 Fact Statistics
                    </h2>

                    <p>
                    💡 Total Facts Viewed:
                    <b>
                    {total_facts}
                    </b>
                    </p>

                    <p>
                    🌐 Different Sources:
                    <b>
                    {unique_sources}
                    </b>
                    </p>

                </div>
                """
            )
        )


        # ----------------------------------------------------
        # SOURCE CHART
        # ----------------------------------------------------

        source_counts = (
            history_df[
                "Source"
            ]
            .value_counts()
            .head(10)
        )


        plt.figure(
            figsize=(9, 5)
        )


        source_counts.plot(
            kind="bar"
        )


        plt.title(
            "Facts by Source"
        )


        plt.xlabel(
            "Source"
        )


        plt.ylabel(
            "Number of Facts"
        )


        plt.xticks(
            rotation=45,
            ha="right"
        )


        plt.tight_layout()


        plt.show()


# ============================================================
# CLEAR HISTORY
# ============================================================

def clear_history(
    button=None
):

    global history_df


    history_df = pd.DataFrame(
        columns=[
            "Date",
            "Time",
            "Fact",
            "Source"
        ]
    )


    history_df.to_csv(
        HISTORY_FILE,
        index=False
    )


    with output:

        clear_output()


        display(
            HTML(
                """
                <div style="
                    background:#ffebee;
                    padding:20px;
                    border-radius:12px;
                ">

                    <h3>
                    🗑️ Fact history cleared.
                    </h3>

                </div>
                """
            )
        )


# ============================================================
# ABOUT PROJECT
# ============================================================

about_button = widgets.Button(
    description="ℹ️ About"
)


def show_about(
    button=None
):

    with output:

        clear_output()


        display(
            HTML(
                """
                <div style="
                    background:#e3f2fd;
                    padding:25px;
                    border-radius:18px;
                ">

                    <h2>
                    🌍 About Random Fact Explorer
                    </h2>

                    <p>
                    Random Fact Explorer is an
                    API-based Python application
                    that fetches interesting facts
                    from an online API.
                    </p>

                    <h3>
                    Technologies Used
                    </h3>

                    <ul>
                        <li>Python</li>
                        <li>Requests</li>
                        <li>JSON API</li>
                        <li>Pandas</li>
                        <li>Matplotlib</li>
                        <li>ipywidgets</li>
                        <li>CSV File Handling</li>
                    </ul>

                    <h3>
                    Main Features
                    </h3>

                    <ul>
                        <li>Random Fact Generation</li>
                        <li>API Integration</li>
                        <li>Favorite Facts</li>
                        <li>Fact History</li>
                        <li>Statistics</li>
                        <li>Data Visualization</li>
                        <li>CSV Storage</li>
                        <li>API Response Time</li>
                    </ul>

                </div>
                """
            )
        )


# ============================================================
# BUTTON EVENTS
# ============================================================

new_fact_button.on_click(
    show_new_fact
)


favorite_button.on_click(
    favorite_fact
)


history_button.on_click(
    show_history
)


statistics_button.on_click(
    show_statistics
)


clear_history_button.on_click(
    clear_history
)


about_button.on_click(
    show_about
)


# ============================================================
# DISPLAY APPLICATION
# ============================================================

display(
    header
)


display(
    HTML(
        """
        <h3 style="
            text-align:center;
        ">
            Discover something interesting!
        </h3>
        """
    )
)


display(
    widgets.HBox(
        [
            new_fact_button,
            favorite_button
        ],
        layout=widgets.Layout(
            justify_content="center",
            margin="15px"
        )
    )
)


display(
    output
)


display(
    status_output
)


display(
    widgets.HBox(
        [
            history_button,
            statistics_button,
            clear_history_button,
            about_button
        ],
        layout=widgets.Layout(
            justify_content="center",
            flex_wrap="wrap",
            margin="15px"
        )
    )
)


print(
    "🌍 Random Fact Explorer started successfully!"
)

print(
    "Click '🔄 New Fact' to fetch a fact from the API."
)

HTML(value='\n    <div style="\n        background:#6a1b9a;\n        color:white;\n        padding:25px;\n    …

Output()

Output()

🌍 Random Fact Explorer started successfully!
Click '🔄 New Fact' to fetch a fact from the API.
